# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

## 1. Method choice and why

**Question shape:** "which pages should a reviewer look at first?" — a ranking problem. Any classifier that outputs a probability can rank; I evaluate the ranking with Precision@K, the same metric as the ML-07 baseline.

**Methods:**
- **Logistic regression** — the readable choice. It learns one weight per feature, so I can check that what it leans on makes sense. It is essentially my baseline rule with learned weights instead of +1 per flag.
- **Gradient boosting (HistGradientBoosting)** — the stronger, non-linear choice, to see whether extra complexity earns its place.

**Features:** the same 23 leakage-safe features built in ML-05 (only the older-30d and prev-30d windows plus static page properties; nothing from the label window).

**Baseline:** the frozen ML-07 rule, recomputed in this notebook run so all numbers come from the same data.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter
print(os.getcwd())

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]

# --- features: the same 23 leakage-safe features as ML-05 ---
X = pd.DataFrame(index=df.index)
for m in ["impressions", "clicks", "sessions"]:
    older = df[f"{m}_90d"] - df[f"{m}_last_30d"] - df[f"{m}_prev_30d"]
    X[f"log_{m}_older_30d"] = np.log1p(older)
    X[f"log_{m}_prev_30d"]  = np.log1p(df[f"{m}_prev_30d"])
X["early_momentum"] = X["log_impressions_prev_30d"] - X["log_impressions_older_30d"]
X["ctr_prev_30d"] = np.where(df["impressions_prev_30d"] > 0,
                             df["clicks_prev_30d"] / df["impressions_prev_30d"].replace(0, 1) * 100, 0)
X["content_age_days"] = df["content_age_days"]
X["has_word_count"] = df["word_count"].notna().astype(int)
X["word_count"] = df["word_count"].fillna(0)
X["has_keyword_data"] = df["search_volume"].notna().astype(int)
X["log_search_volume"] = np.log1p(df["search_volume"].fillna(0))
X["competition"] = df["competition"].fillna(0)
X["cpc"] = df["cpc"].fillna(0)
X = pd.concat([X, pd.get_dummies(df[["content_type", "main_intent"]].fillna("unknown"), dtype=int)], axis=1)

# --- baseline: the frozen ML-07 rule, recomputed in this run ---
prev  = df["impressions_prev_30d"]
older = df["impressions_90d"] - df["impressions_last_30d"] - prev
ctr_prev = np.where(prev > 0, df["clicks_prev_30d"] / prev.replace(0, 1) * 100, 0)
momentum = np.log1p(prev) - np.log1p(older)
flags = pd.DataFrame({
    "visible_page":   prev >= 200,
    "young_page":     df["content_age_days"] <= 180,
    "low_ctr_page":   (ctr_prev < 0.1) & (prev >= 200),
    "recent_spike":   momentum > 0.3,
    "keyword_target": df["search_volume"].notna(),
})
baseline_score = (flags.sum(axis=1) + np.log1p(prev).rank(pct=True) * 0.5).values

print("features:", X.shape, "| base rate:", round(y.mean(), 3))

/content/flyrank-ml-internship-starter
/content/flyrank-ml-internship-starter
features: (30000, 23) | base rate: 0.542


## 2. Split design

## 2. Split design

**GroupKFold(5) by client.** Pages from one client share hidden character (site, niche, update habits). A random split lets the model memorise clients — in ML-05 that inflated AUC from 0.68 to 0.79. FlyRank's real use is ranking pages for clients the model has not seen, so every test fold contains only unseen clients.

**Out-of-fold scores:** each page is scored by the model trained on the other four folds. That gives an honest score for all 30,000 pages, so the model and the rule are compared on exactly the same rows and the same metric.

**Not time-aware:** the CSV is a single snapshot (one label window), so a past→future split is not possible here; the time logic is handled inside the features (only data from before the label window).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

oof = {"logreg": np.zeros(len(df)), "hgb": np.zeros(len(df))}   # out-of-fold scores
fold_of = np.zeros(len(df), dtype=int)

for i, (train_idx, test_idx) in enumerate(GroupKFold(n_splits=5).split(X, y, groups)):
    fold_of[test_idx] = i
    logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    logreg.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof["logreg"][test_idx] = logreg.predict_proba(X.iloc[test_idx])[:, 1]

    hgb = HistGradientBoostingClassifier(random_state=42)
    hgb.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof["hgb"][test_idx] = hgb.predict_proba(X.iloc[test_idx])[:, 1]

    print(f"fold {i}: {len(test_idx)} test rows, {groups.iloc[test_idx].nunique()} unseen clients")

# sanity: no client appears in two folds
print("clients split cleanly:", pd.Series(fold_of).groupby(groups.values).nunique().max() == 1)

fold 0: 7008 test rows, 1 unseen clients
fold 1: 5731 test rows, 7 unseen clients
fold 2: 5753 test rows, 8 unseen clients
fold 3: 5755 test rows, 8 unseen clients
fold 4: 5753 test rows, 8 unseen clients
clients split cleanly: True


## 3. Train + compare vs my baseline

## 3. Train + compare vs my baseline

Same 30,000 rows, same label, same metric (Precision@K), same client-grouped evaluation. The random row is the base rate: what picking pages blindly would give on average.

**Observed:**
- **Logistic regression beats the rule at every K:** P@50 0.88 vs 0.80, P@500 0.83 vs 0.78. It wins in 4 of 5 folds; in fold 2 (lowest base rate, 0.38) the rule is ahead (0.80 vs 0.74), so the gain is directional, not guaranteed for every client group.
- **Gradient boosting has the best AUC (0.698) but the worst top of the list** (P@50 0.62, below the rule). AUC scores the whole ranking; a reviewer only reads the top. For this question the top is what matters, so logistic regression is the model I keep.
- More complexity did not earn its place here: the simpler, readable model is the better tool for "which pages first".

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, k):
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return y.iloc[top].mean()

methods = {"baseline rule (ML-07)": baseline_score, "logistic regression": oof["logreg"],
           "gradient boosting (HGB)": oof["hgb"]}
rows = [["random (expected)"] + [round(y.mean(), 2)] * 4 + [0.5]]
for name, s in methods.items():
    rows.append([name] + [round(precision_at_k(s, k), 2) for k in (10, 20, 50, 500)]
                + [round(roc_auc_score(y, s), 3)])
table = pd.DataFrame(rows, columns=["method", "P@10", "P@20", "P@50", "P@500", "AUC"])
print(table.to_string(index=False))

# stability: P@50 inside each fold (each fold = clients the model never saw)
per_fold = []
for i in range(5):
    m = fold_of == i
    p = lambda s: y[m].values[np.argsort(-np.asarray(s)[m], kind="stable")[:50]].mean()
    per_fold.append([i, round(y[m].mean(), 2), p(baseline_score), p(oof["logreg"]), p(oof["hgb"])])
print(pd.DataFrame(per_fold, columns=["fold", "base_rate", "rule", "logreg", "hgb"]).to_string(index=False))

                 method  P@10  P@20  P@50  P@500   AUC
      random (expected)  0.54  0.54  0.54   0.54 0.500
  baseline rule (ML-07)  0.60  0.65  0.80   0.78 0.650
    logistic regression  0.80  0.85  0.88   0.83 0.673
gradient boosting (HGB)  0.50  0.65  0.72   0.67 0.698
 fold  base_rate  rule  logreg  hgb
    0       0.49  0.78    0.84 0.82
    1       0.65  0.82    0.88 0.72
    2       0.38  0.80    0.74 0.68
    3       0.62  0.74    0.86 0.74
    4       0.58  0.82    0.96 0.84


## 4. Errors and interpretation

## 4. Errors and interpretation

**What it leans on (coefficients, + = more likely to decline):**
- `log_impressions_prev_30d` +0.62 and `log_clicks_prev_30d` −0.56: many impressions but few clicks → decline. This is the same signal as the rule's `low_ctr_page`, learned with weights.
- `log_impressions_older_30d` +0.53: pages that already had real demand two months ago.
- `content_age_days` −0.31: younger pages decline more — the same idea as `young_page`.
- `early_momentum` +0.24: a recent rise tends to fall back, but only mildly (in ML-07 the spike flag was the weakest reason).
- The top features match the rule's reasoning and none is "suspiciously perfect", which supports that no leakage slipped in.

**Is it just one client?** The raw top-50 comes from only 2 clients (39 + 11), so I re-ranked with at most 5 pages per client: logistic regression keeps P@50 0.92 across 14 clients, while the rule drops to 0.72. The model's advantage survives the client cap, so it is not only a one-client effect.

**The misses:** all 6 wrong pages in the top-50 are stable or up. Their prev-30d impressions were close to or above their older-30d impressions, so nothing in the past window distinguished them from pages that fell — they look like decliners but plateaued.

**Where it is weak:** by traffic level, the model separates well only among low-traffic pages (AUC 0.71); among mid and high traffic it is close to random (AUC 0.57–0.63). Much of its skill is "which pages are big enough to fall", not "which big page will fall". Treat it as decision support for ordering a review queue, not a forecast.

**Reproducibility:** random_state=42 for gradient boosting; scikit-learn 1.8.0. Tree models can shift by a few points between library versions.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1) what does logistic regression lean on? (coefficients on standardised features)
final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X, y)
coef = pd.Series(final[-1].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("top coefficients (+ = more likely to decline):")
print(coef.head(6).round(2).to_string())

# 2) is the top-50 just one client?
top50 = np.argsort(-oof["logreg"], kind="stable")[:50]
print("\nclients in logreg top-50:", groups.iloc[top50].nunique(),
      groups.iloc[top50].str[-4:].value_counts().to_dict())

def capped_p50(scores, cap=5):          # at most 5 pages per client
    d = pd.DataFrame({"s": scores, "c": groups, "y": y}).sort_values("s", ascending=False, kind="stable")
    d = d.groupby("c").head(cap).head(50)
    return float(round(d["y"].mean(), 2)), int(d["c"].nunique())
for name, s in methods.items():
    print(f"{name:26s} P@50 with max 5 per client: {capped_p50(s)}  (precision, clients)")

# 3) the misses: logreg top-50 pages that did NOT decline (outcome columns shown only for checking)
miss = df.iloc[top50][y.iloc[top50].values == 0]
print("\nmisses in top-50:", len(miss))
print(miss.assign(client=miss["client_id"].str[-4:],
                  older_30d=miss["impressions_90d"] - miss["impressions_last_30d"] - miss["impressions_prev_30d"])
      [["client", "older_30d", "impressions_prev_30d", "impressions_last_30d", "content_age_days", "trend_direction"]]
      .to_string(index=False))

# 4) where is it weak? AUC by traffic level (prev 30d impressions, quartiles)
d = pd.DataFrame({"y": y, "s": oof["logreg"],
                  "traffic": pd.qcut(prev.rank(method="first"), 4, labels=["low", "mid-low", "mid-high", "high"])})
print("\n", d.groupby("traffic", observed=True).apply(
    lambda g: pd.Series({"base_rate": g["y"].mean(), "auc": roc_auc_score(g["y"], g["s"])})).round(3).to_string())

top coefficients (+ = more likely to decline):
log_impressions_prev_30d     0.62
log_clicks_prev_30d         -0.56
log_impressions_older_30d    0.53
content_age_days            -0.31
early_momentum               0.24
log_sessions_prev_30d       -0.21

clients in logreg top-50: 2 {'0f77': 39, '27de': 11}
baseline rule (ML-07)      P@50 with max 5 per client: (0.72, 12)  (precision, clients)
logistic regression        P@50 with max 5 per client: (0.92, 14)  (precision, clients)
gradient boosting (HGB)    P@50 with max 5 per client: (0.74, 12)  (precision, clients)

misses in top-50: 6
client  older_30d  impressions_prev_30d  impressions_last_30d  content_age_days trend_direction
  27de       1432                  9527                  8401               236          stable
  0f77       2568                  2690                  2457               118          stable
  0f77       3098                  3828                  3899                98          stable
  27de       2560         

/tmp/ipykernel_2172/492433843.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\n", d.groupby("traffic", observed=True).apply(


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.